In [ ]:
"""MiniPong — the world of the re-recorded Lecture 3.

A 32x32 RGB paddle game:
  * a bright paddle at the bottom, moved by 3 actions (left / stay / right)
  * a soft glowing ball bouncing off the left, right, top and bottom walls,
    and bending off the paddle
  * the ball's VELOCITY is invisible in a single frame — the memory story
  * dark background: the objects carry all the signal (easy, honest training)

Two properties matter for learnability, and we chose both deliberately:
  * DETERMINISTIC physics — no randomness after reset, so a predictor can in
    principle be exact (genuine randomness needs richer heads; a later lecture)
  * CONTINUOUS motion with anti-aliased rendering — the ball's sub-pixel
    position shows up smoothly in pixel intensities, so nearby states get
    nearby codes. A world snapped to integer pixels creates isolated islands
    of codes, and a predicted code that lands between islands paints nothing.

No reward is defined: this lecture builds ONLY the world model.
"""
import numpy as np


class MiniPong:
    SIZE = 32
    ACTIONS = 3            # 0 = paddle left, 1 = stay, 2 = paddle right
    PADDLE_W = 8
    PADDLE_Y = 29          # paddle row (2 px tall: rows 29-30)
    PADDLE_SPEED = 2
    BG = np.array([0.05, 0.05, 0.08], np.float32)
    WALL = np.array([0.35, 0.35, 0.40], np.float32)
    PADDLE = np.array([0.20, 0.85, 0.80], np.float32)   # bright teal
    BALL = np.array([1.00, 0.85, 0.30], np.float32)     # bright warm yellow

    _YY, _XX = np.mgrid[0:32, 0:32].astype(np.float32)

    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.px = float(self.rng.integers(6, self.SIZE - 6 - self.PADDLE_W))  # paddle left edge
        self.bx = float(self.rng.uniform(6, self.SIZE - 6))
        self.by = float(self.rng.uniform(4, 12))
        # continuous, deterministic velocity: fixed after reset, ~1 px per step
        self.vx = float(self.rng.choice([-1, 1]) * self.rng.uniform(0.55, 0.95))
        self.vy = float(self.rng.uniform(0.55, 0.95))
        self.t = 0
        return self.observe()

    def observe(self):
        img = np.tile(self.BG, (self.SIZE, self.SIZE, 1)).astype(np.float32)
        img[0, :] = self.WALL
        img[:, 0] = self.WALL
        img[:, -1] = self.WALL
        p = int(round(self.px))
        img[self.PADDLE_Y:self.PADDLE_Y + 2, p:p + self.PADDLE_W] = self.PADDLE
        # the ball is a soft glow: its sub-pixel position shows in the intensities
        g = 1.5 * np.exp(-(((self._XX - self.bx) ** 2 + (self._YY - self.by) ** 2)
                           / (2 * 1.15 ** 2)))[..., None]
        img = np.clip(img + g * self.BALL, 0.0, 1.0).astype(np.float32)
        return img

    def step(self, action):
        # paddle
        self.px += (action - 1) * self.PADDLE_SPEED
        self.px = float(np.clip(self.px, 1, self.SIZE - 1 - self.PADDLE_W))
        # ball — continuous, deterministic motion with mirror reflections
        self.bx += self.vx
        self.by += self.vy
        if self.bx < 2.0:                        # left wall
            self.bx = 4.0 - self.bx; self.vx = abs(self.vx)
        if self.bx > self.SIZE - 3.0:            # right wall
            self.bx = 2 * (self.SIZE - 3.0) - self.bx; self.vx = -abs(self.vx)
        if self.by < 2.0:                        # top wall
            self.by = 4.0 - self.by; self.vy = abs(self.vy)
        # paddle bounce — and the key pressed at contact deterministically bends the ball
        if self.vy > 0 and self.by >= self.PADDLE_Y - 1.5:
            if self.px - 1 <= self.bx <= self.px + self.PADDLE_W:
                self.by = 2 * (self.PADDLE_Y - 1.5) - self.by
                self.vy = -abs(self.vy)
                if action == 0:
                    self.vx = -0.9
                elif action == 2:
                    self.vx = 0.9
        # bottom wall: the ball bounces (no random respawns — the world stays
        # perfectly deterministic, which is exactly what a first world model needs)
        if self.by > self.SIZE - 2.5:
            self.by = 2 * (self.SIZE - 2.5) - self.by; self.vy = -abs(self.vy)
        self.t += 1
        return self.observe(), self.t >= 120


if __name__ == "__main__":
    env = MiniPong(seed=0)
    obs = env.reset()
    rng = np.random.default_rng(0)
    n_bounce = 0
    for t in range(240):
        vy_before = env.vy
        obs, done = env.step(int(rng.integers(0, 3)))
        if vy_before > 0 and env.vy < 0:
            n_bounce += 1
    print(f"smoke test: 240 steps, {n_bounce} upward bounces, ball at ({env.bx:.1f},{env.by:.1f}), frame {obs.shape}")

In [ ]:
import math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import rcParams

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PAPER, INK, MUTED = "#FBF9F1", "#16130D", "#6D665A"
TEAL, GOLD, CLAY = "#2E8F8F", "#DD9F3E", "#C96442"
rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "serif", "axes.titlesize": 13, "axes.titleweight": "bold",
})

In [ ]:
env = MiniPong(seed=SEED)
EPISODES, T = 200, 120
frames = np.zeros((EPISODES, T, 32, 32, 3), np.float32)
actions = np.zeros((EPISODES, T), np.int64)
rng = np.random.default_rng(SEED)
for e in range(EPISODES):
    obs = env.reset()
    a = int(rng.integers(0, 3))
    for t in range(T):
        if rng.random() < 0.25:            # switch keys only occasionally
            a = int(rng.integers(0, 3))
        frames[e, t], actions[e, t] = obs, a
        obs, done = env.step(a)
print(f"dataset: {EPISODES} episodes × {T} steps = {EPISODES*T:,} frames "
      f"({frames.nbytes/1e6:.0f} MB)")

fig, axes = plt.subplots(1, 8, figsize=(13, 1.9))
for i in range(8):
    axes[i].imshow(frames[i, i * 13]); axes[i].axis("off")
fig.suptitle("MiniPong — eight raw observations from the dataset", y=1.12)
plt.tight_layout(); plt.savefig("plot_env_frames.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
Z = 12

class VNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(                       # 3 x 32 x 32
            nn.Conv2d(3, 16, 4, 2, 1), nn.ReLU(),      # 16 x 16 x 16
            nn.Conv2d(16, 32, 4, 2, 1), nn.ReLU(),     # 32 x 8 x 8
            nn.Flatten())                               # 2048
        self.mu = nn.Linear(2048, Z)
        self.logvar = nn.Linear(2048, Z)
        self.fc = nn.Linear(Z, 2048)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 4, 2, 1), nn.Sigmoid())

    def encode(self, x):
        h = self.enc(x.permute(0, 3, 1, 2))
        return self.mu(h), self.logvar(h).clamp(-6, 2)

    def decode(self, z):
        return self.dec(self.fc(z).view(-1, 32, 8, 8)).permute(0, 2, 3, 1)

vnet = VNet().to(device)
opt = torch.optim.Adam(vnet.parameters(), lr=1e-3)
flat = torch.tensor(frames.reshape(-1, 32, 32, 3))
# train on CONSECUTIVE PAIRS of frames — we need them for the smoothness term below
pair_idx = torch.tensor([e * T + t for e in range(EPISODES) for t in range(T - 1)])
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(pair_idx),
                                     batch_size=256, shuffle=True)
t0 = time.time()
for epoch in range(20):
    tot = 0.0
    for (bi,) in loader:
        x = flat[bi].to(device)             # frame t
        xn = flat[bi + 1].to(device)        # frame t+1 (same episode)
        mu, logvar = vnet.encode(x)
        mun, _ = vnet.encode(xn)
        # deterministic code: z IS mu. The same frame always gets the same code — network
        # M can only predict the next code as exactly as V computes it, so the code must
        # be an exact function of the frame, not a noisy sample around it.
        z = mu + 0.1 * torch.randn_like(mu)  # fuzz for the DECODER only: predicted codes
                                             # will land NEAR real ones, so train it there
        xhat = vnet.decode(z)
        # the ball is a handful of pixels out of 1,024 — weight RED pixels up (only
        # the ball is red), or compression happily drops the most important thing
        w = 1.0 + 40.0 * x[..., :1]        # weight by REDNESS: only the ball is red
        recon = (w * F.binary_cross_entropy(xhat, x, reduction="none")).sum() / len(x)
        # SMOOTHNESS: the world moves one pixel at a time, so consecutive frames must
        # get nearby codes. Without this, V is free to scatter neighbouring states to
        # far-apart codes — decodable, but impossible for network M to predict.
        smooth = ((mu - mun) ** 2).sum(1).mean()
        # plus a light pull toward zero to keep the code scale bounded
        loss = recon + 1.0 * smooth + 0.01 * (mu ** 2).sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(x)
    print(f"epoch {epoch+1:2d}  loss {tot/len(flat):8.1f}")
print(f"V trained in {time.time()-t0:.0f}s   ({sum(p.numel() for p in vnet.parameters()):,} parameters)")

In [ ]:
vnet.eval()
with torch.no_grad():
    test = flat[5000:24000:2713][:7].to(device)
    mu, _ = vnet.encode(test)
    rec = vnet.decode(mu)
fig, axes = plt.subplots(2, 7, figsize=(12, 3.6))
for i in range(7):
    axes[0, i].imshow(test[i].cpu()); axes[0, i].axis("off")
    axes[1, i].imshow(rec[i].cpu().clamp(0, 1)); axes[1, i].axis("off")
axes[0, 0].set_title("original frame", loc="left", color=TEAL)
axes[1, 0].set_title("redrawn from the 12-number code", loc="left", color=CLAY)
fig.suptitle("Network V — 3,072 pixels → 12 numbers → 3,072 pixels", y=1.02)
plt.tight_layout(); plt.savefig("plot_v_recon.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
H = 128

class MNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(Z + 3, H, batch_first=True)   # network 1: memory update
        self.head = nn.Sequential(                       # network 2: prediction
            nn.Linear(H, 128), nn.ELU(), nn.Linear(128, Z))

    def forward(self, z_seq, a_seq, h0=None):
        inp = torch.cat([z_seq, a_seq], -1)
        out, hN = self.gru(inp, h0)
        return self.head(out), hN

with torch.no_grad():
    zs = []
    for i in range(0, len(flat), 4096):
        mu, _ = vnet.encode(flat[i:i+4096].to(device))
        zs.append(mu.cpu())
    zs = torch.cat(zs).view(EPISODES, T, Z)
# normalise each code dimension to unit scale for M's training — otherwise the
# dimensions describing the (big, busy) paddle dominate the loss and the (tiny)
# ball's dimensions get ignored. We undo this before decoding.
Z_MEAN = zs.reshape(-1, Z).mean(0)
Z_STD = zs.reshape(-1, Z).std(0).clamp_min(1e-4)
zs = (zs - Z_MEAN) / Z_STD
a1h = F.one_hot(torch.tensor(actions), 3).float()

mnet = MNet().to(device)
opt = torch.optim.Adam(mnet.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(25):
    perm, tot = torch.randperm(EPISODES), 0.0
    for i in range(0, EPISODES, 16):
        idx = perm[i:i+16]
        z = zs[idx].to(device); a = a1h[idx].to(device)
        # two robustness tricks that make the closed loop survivable:
        #   (a) the head predicts the CHANGE in the code, not the code itself —
        #       under uncertainty the safest change is "nothing moves", which keeps
        #       the ball painted instead of fading it toward an invisible average;
        #   (b) we add a little noise to the input codes, so the network practices
        #       CORRECTING a slightly-wrong code instead of amplifying the error.
        z_in = z[:, :-1] + 0.05 * torch.randn_like(z[:, :-1])
        pred, _ = mnet(z_in, a[:, :-1])
        loss = F.mse_loss(z_in + pred, z[:, 1:])
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(idx)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1:2d}  prediction error {tot/EPISODES:8.4f}")
print(f"M trained in {time.time()-t0:.0f}s   ({sum(p.numel() for p in mnet.parameters()):,} parameters)")

In [ ]:
# fine-tune on short CLOSED-LOOP rollouts: the model must survive eating its own
# predictions for 5 steps. Crucially, we also DECODE each predicted code and demand
# that the painted frame matches the real one — this keeps the imagined ball a
# sharp dot instead of letting it diffuse into invisible mist.
Z_MEAN_d, Z_STD_d = Z_MEAN.to(device), Z_STD.to(device)
for p in vnet.parameters():
    p.requires_grad_(False)
t0 = time.time()
STAGES = [(5, 1e-3, 25), (15, 3e-4, 15)]   # curriculum: survive 5 steps, then 15
for K, lr, n_epochs in STAGES:
  for g in opt.param_groups:
      g["lr"] = lr
  for epoch in range(n_epochs):
    perm, tot = torch.randperm(EPISODES), 0.0
    for i in range(0, EPISODES, 16):
        idx = perm[i:i+16]
        z = zs[idx].to(device); a = a1h[idx].to(device)
        x = torch.tensor(frames[idx.numpy()]).to(device)
        t_start = int(torch.randint(4, T - K - 1, (1,)))
        _, h = mnet(z[:, :t_start], a[:, :t_start])
        zc = z[:, t_start:t_start+1]
        loss = 0.0
        for k in range(K):
            pred, h = mnet(zc, a[:, t_start+k:t_start+k+1], h)
            znext = zc + pred                                # code + predicted change
            loss = loss + F.mse_loss(znext[:, 0], z[:, t_start+k+1])
            xk = x[:, t_start+k+1]
            xhat = vnet.decode(znext[:, 0] * Z_STD_d + Z_MEAN_d)
            wpix = 1.0 + 40.0 * xk[..., :1]
            loss = loss + 0.002 * (wpix * F.binary_cross_entropy(xhat, xk, reduction="none")).sum() / len(idx)
            zc = znext
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(idx)
    if (epoch + 1) % 5 == 0:
        print(f"rollout K={K:2d} epoch {epoch+1:2d}  loss {tot/EPISODES:8.3f}")
print(f"closed-loop fine-tune in {time.time()-t0:.0f}s")
torch.save({"vnet": vnet.state_dict(), "mnet": mnet.state_dict(),
            "z_mean": Z_MEAN, "z_std": Z_STD}, "pong_wm.pt")

In [ ]:
mnet.eval()
with torch.no_grad():
    z = zs[:100].to(device); a = a1h[:100].to(device)
    pred, _ = mnet(z[:, :-1], a[:, :-1])
    err = ((z[:, :-1] + pred - z[:, 1:]) ** 2).mean(-1).cpu().numpy()   # (100, T-1)
fig, ax = plt.subplots(figsize=(8.2, 3.6))
steps = np.arange(1, 16)
ax.plot(steps, err[:, :15].mean(0), color=TEAL, lw=2.5, marker="o", ms=5)
ax.set_xticks(steps)
ax.set_xlabel("timestep being predicted")
ax.set_ylabel("prediction error")
ax.set_title("The memory needs exactly two frames — then prediction snaps into place")
plt.tight_layout(); plt.savefig("plot_memory_snap.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"prediction error at t=1: {err[:,0].mean():.4f}   at t=3: {err[:,2].mean():.4f}   "
      f"at t=10: {err[:,9].mean():.4f}")

In [ ]:
env = MiniPong(seed=8)
obs = env.reset()
real = [obs]
script = [2]*6 + [0]*7 + [1]*4 + [2]*5
for a in [1, 1] + script:
    obs, _ = env.step(a)
    real.append(obs)
with torch.no_grad():
    real_t = torch.tensor(np.stack(real), dtype=torch.float32).to(device)
    mu, _ = vnet.encode(real_t)
    zt = (mu - Z_MEAN.to(device)) / Z_STD.to(device)
    acts = [1, 1] + script
    a_all = F.one_hot(torch.tensor([acts]), 3).float().to(device)
    pred_seq, _ = mnet(zt[None, :-1], a_all[:, :len(zt) - 1])
    znext = zt[None, :-1] + pred_seq                  # predicted next codes
    painted = vnet.decode(znext[0] * Z_STD.to(device) + Z_MEAN.to(device)).cpu().numpy()

offs = 2                                              # skip the 2 warm-up steps
glyph = {0: "left", 1: "stay", 2: "right"}
show = list(range(0, len(script), 3))
fig, axes = plt.subplots(2, len(show), figsize=(13, 4.0))
for i, t in enumerate(show):
    axes[0, i].imshow(np.clip(real[offs + 1 + t], 0, 1)); axes[0, i].axis("off")
    axes[0, i].set_title(f"step {t+1} · key {glyph[script[t]]}", fontsize=8, color=MUTED)
    axes[1, i].imshow(np.clip(painted[offs + t], 0, 1)); axes[1, i].axis("off")
fig.text(0.075, 0.68, "what really\nhappened", fontsize=11, color=TEAL, ha="right", fontweight="bold")
fig.text(0.075, 0.27, "painted by the model\n— before it happened", fontsize=11, color=CLAY, ha="right", fontweight="bold")
fig.suptitle("At every step, the network paints the NEXT frame before the game shows it", y=0.99)
plt.subplots_adjust(left=0.10, top=0.85, bottom=0.03, wspace=0.06, hspace=0.16)
plt.savefig("plot_pred_next.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
def dream(actions_list, warm_ep=2, warm_steps=3):
    """Run the model closed-loop with a given list of actions; return decoded frames.
    Alignment matters: the warm-up consumes codes 0..warm-1, so the first input of
    the closed loop is the REAL code at index `warm_steps` — never a code the GRU
    has already eaten."""
    with torch.no_grad():
        _, h = mnet(zs[warm_ep:warm_ep+1, :warm_steps].to(device),
                    a1h[warm_ep:warm_ep+1, :warm_steps].to(device))
        z = zs[warm_ep:warm_ep+1, warm_steps:warm_steps+1].to(device)
        out = []
        for a in actions_list:
            ah = F.one_hot(torch.tensor([[a]]), 3).float().to(device)
            pred, h = mnet(z, ah, h)
            z = z + pred[:, -1:]                             # code + predicted change
            z_raw = z[:, 0] * Z_STD.to(device) + Z_MEAN.to(device)   # back to V's scale
            out.append(vnet.decode(z_raw)[0].cpu())
        return torch.stack(out)

# a scripted "player": hold right, hold left, then stay — and run the SAME keys
# through the real engine, so we can put reality and the dream side by side
script = [2]*5 + [0]*5 + [1]*4
env = MiniPong(seed=11)
obs = env.reset()
real = [obs]
for a in [1, 1, 1] + script:                      # 3 warm-up frames, then the script
    obs, _ = env.step(a)
    real.append(obs)
real_t = torch.tensor(np.stack(real), dtype=torch.float32).to(device)
with torch.no_grad():                             # dream from the same 3-frame warm-up
    mu, _ = vnet.encode(real_t)
    zt = ((mu - Z_MEAN.to(device)) / Z_STD.to(device))
    _, h = mnet(zt[None, :3], F.one_hot(torch.tensor([[1, 1, 1]]), 3).float().to(device))
    z = zt[None, 3:4]
    dreamed = []
    for a in script:
        pred, h = mnet(z, F.one_hot(torch.tensor([[a]]), 3).float().to(device), h)
        z = z + pred[:, -1:]
        dreamed.append(vnet.decode(z[:, 0] * Z_STD.to(device) + Z_MEAN.to(device))[0].cpu())

fig, axes = plt.subplots(2, 7, figsize=(13, 4.0))
show = list(range(0, len(script), 2))
glyph = {0: "left", 1: "stay", 2: "right"}
for i, t in enumerate(show):
    axes[0, i].imshow(np.clip(real[4 + t], 0, 1)); axes[0, i].axis("off")
    axes[0, i].set_title(f"step {t+1} · key {glyph[script[t]]}", fontsize=8, color=MUTED)
    axes[1, i].imshow(dreamed[t].clamp(0, 1)); axes[1, i].axis("off")
fig.text(0.075, 0.68, "real game\n(engine on)", fontsize=11, color=TEAL, ha="right", fontweight="bold")
fig.text(0.075, 0.27, "the dream\n(engine off)", fontsize=11, color=CLAY, ha="right", fontweight="bold")
fig.suptitle("The same keys, two worlds — the bottom one is the network's imagination", y=0.99)
plt.subplots_adjust(left=0.10, top=0.85, bottom=0.03, wspace=0.06, hspace=0.16)
plt.savefig("plot_dream_play.png", dpi=150, bbox_inches="tight"); plt.show()